In [ ]:
import sys

import polars as pl
import torch


from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule
from modeling_module.data_loader.MultiPartExoDataModule import MultiPartExoDataModule

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

save_dir = DIR + 'fit/20251109_LTB'


In [ ]:
target_dyn_demand_monthly = pl.read_parquet(DIR + 'target_dyn_demand_monthly.parquet').sort(['oper_part_no', 'demand_dt'])
target_dyn_demand_monthly = (target_dyn_demand_monthly.group_by('oper_part_no', maintain_order = True).map_groups(lambda g: g.with_columns(pl.arange(1, len(g) + 1).alias('sequence'))))

filtered_target = (target_dyn_demand_monthly
                    .group_by('oper_part_no')
                    .agg(pl.col('sequence').max().alias('sequence_max'))
                    .filter(pl.col('sequence_max') > 43)
                    .select('oper_part_no')
                   ) # seq Q75

target_dyn_demand_monthly = (target_dyn_demand_monthly
                                .join(filtered_target, on = 'oper_part_no', how = 'right')
                                .select(['oper_part_no', 'demand_dt', 'sequence', 'demand_qty'])
                             )
target_dyn_demand_monthly


In [ ]:
plan_yyyymm = 201801
lookback = 12
horizon = 3

# data_module = MultiPartDataModule(
#     target_dyn_demand_monthly,
#     lookback = lookback,
#     horizon = horizon,
#     batch_size = 64,
#     val_ratio = 0.2,
#     is_running = False
# )
# train_loader = data_module.get_train_loader()
# val_loader = data_module.get_val_loader()

data_module = MultiPartExoDataModule(
    target_dyn_demand_monthly,
    lookback = lookback,
    horizon = horizon,
    batch_size = 64,
    val_ratio = 0.2,
    is_running = False
)

train_loader = data_module.get_train_loader()
val_loader = data_module.get_val_loader()

In [ ]:
from modeling_module.training.model_trainers.total_train import run_total_train_monthly

model_dict = run_total_train_monthly(train_loader, val_loader, lookback = lookback, horizon = horizon)

In [ ]:
from modeling_module.models.Titan.common.configs import TitanConfig
from modeling_module.utils.exogenous_utils import calendar_sin_cos
from modeling_module.models.PatchTST.common.configs import PatchTSTConfigMonthly
from modeling_module.models.PatchMixer.common.configs import PatchMixerConfigMonthly, PatchMixerConfig

save_dir = DIR + 'fit'
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

pm_base_config = PatchMixerConfig(
        lookback = lookback,
        horizon = horizon,
        device = device,
        loss_mode = 'point',
        point_loss = 'mae'
    )

pm_quantile_config = PatchMixerConfig(
    lookback = lookback,
        horizon = horizon,
    device = device,
    loss_mode = 'quantile',
    quantiles = (0.1, 0.5, 0.9)
)
ti_config = TitanConfig(
        lookback=lookback,
        horizon=horizon,
        input_dim=1,
        d_model=256,
        n_layers=3,
        n_heads=4,
        d_ff=512,
        dropout=0.1,
        contextual_mem_size=256,
        persistent_mem_size=64,
        use_exogenous=True, exo_dim=2,     # month sin/cos
        final_clamp_nonneg=True,
    )

pt_config = PatchTSTConfigMonthly(
        device = device,
        loss_mode = 'auto',
        quantiles = (0.1, 0.5, 0.9)
    )

cfg_map = {
    "PatchMixer Base": pm_base_config,
    "PatchMixer Quantile": pm_quantile_config,
    # "Titan Base": ti_config,
    # "Titan LMM": ti_config,
    # "Titan Seq2Seq": ti_config,
    # "PatchTST Base": pt_config,
    # "PatchTST Quantile": pt_config
}



In [ ]:

save_dir = DIR + 'fit/20251115_LTB'

models_only = {
    name: (pack["model"] if isinstance(pack, dict) and "model" in pack else pack)
    for name, pack in model_dict.items()
}

# cfg_by_name도 맞춰서 준비(필요하면)
cfg_by_name = {
    name: (pack.get("cfg") if isinstance(pack, dict) else None)
    for name, pack in model_dict.items()
}

builder_key_by_name = {
  "PatchMixer Base": "patchmixer_base",
  "PatchMixer Quantile": "patchmixer_quantile",
  # "Titan Base": "titan_base",
  # "Titan LMM": "titan_lmm",
  # "Titan Seq2Seq": "titan_seq2seq",
  # "PatchTST Base": "patchtst_base",
  # "PatchTST Quantile": "patchtst_quantile",
}
save_index = save_model_dict(
    models_only,
    save_dir,
    cfg_by_name = cfg_by_name,
    builder_key_by_name=builder_key_by_name)



In [ ]:
# Load
from modeling_module.models.model_builder import (
    build_patch_mixer_quantile,
    build_patchTST_base, build_patchTST_quantile, build_patch_mixer_base, build_titan_base, build_titan_lmm,
    build_titan_seq2seq,
)

builders = {
    "patchmixer_base": lambda cfg: build_patch_mixer_base(cfg or PatchMixerConfig()),
    "patchmixer_quantile": lambda cfg: build_patch_mixer_quantile(cfg or PatchMixerConfig()),
    # "titan_base": lambda cfg: build_titan_base(cfg or TitanConfig()),
    # "titan_lmm": lambda cfg: build_titan_lmm(cfg or TitanConfig()),
    # "titan_seq2seq": lambda cfg: build_titan_seq2seq(cfg or TitanConfig()),
    # "patchtst_base": lambda cfg: build_patchTST_base(cfg or PatchTSTConfigMonthly()),
    # "patchtst_quantile": lambda cfg: build_patchTST_quantile(cfg or PatchTSTConfigMonthly()),
}
loaded = load_model_dict(save_dir, builders, device = device)


In [ ]:
%load_ext autoreload
%autoreload 2

import importlib, modeling_module.utils.plot_utils as pu
import modeling_module.training.forecaster as fo
importlib.reload(pu)
importlib.reload(fo)

def my_exo_cb(start_idx: int, Hm: int, device="cuda" if torch.cuda.is_available() else "cpu"):
    # exo_dim = 2 (sin, cos)
    return fo.make_calendar_exo(start_idx, Hm, period=12, device=device)

pu.plot_120m(
    models=loaded,           # {"PatchMixer": pm_model, "Titan": ti_model, ...}
    loader=val_loader,       # (xb, yb[, part_ids])
    device="cuda" if torch.cuda.is_available() else "cpu",
    mode="val",              # ← 검증 모드
    max_plots=5,
    out_dir=None,
    show=True,
    future_exo_cb=my_exo_cb
)